# Continuation of 05_train_model.ipynb notebook
- Discovered contamination between model iterations.
- Clean new notebook for experimentation purely with transformer model.
- Reusable functions to ensure clean runs and development

# Load data and artefacts

In [2]:
# Imports
import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import yaml
from minio import Minio


In [3]:
# Load config from yaml
CONFIG_FILE = Path("../config/config.yaml")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Missing config file: {CONFIG_FILE.resolve()}")

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Config loaded")

Config loaded


In [18]:
# Read .env for credentials
def load_env_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path.resolve()}")

    env = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip()
    return env

# Env file location in repo
ENV_FILE = Path(f"{cfg['storage']['minio']['env_file']}")
# Storage endpoint and config
MINIO_ENDPOINT = cfg["storage"]["minio"]["endpoint"]
MINIO_SECURE = cfg["storage"]["minio"]["secure"]

# Authenticate to file storage list buckets to confirm connection
env = load_env_file(ENV_FILE)
client = Minio(
    MINIO_ENDPOINT,
    access_key=env["MINIO_ROOT_USER"],
    secret_key=env["MINIO_ROOT_PASSWORD"],
    secure=MINIO_SECURE,
)

print("Connected buckets:", [b.name for b in client.list_buckets()])

Connected buckets: ['data-profile-test', 'incident-pipeline', 'incident-pipeline-test', 'mlflow-artifacts']


In [19]:
# Load latest gold train, valid and test data from storage
gold_prefix_root = cfg["storage"]["minio"]["gold_training_prefix_root"].rstrip("/")
test_bucket = "incident-pipeline-test"

# Find latest timestamped gold run
run_tags = sorted({
    obj.object_name[len(gold_prefix_root) + 1:].split("/", 1)[0]
    for obj in client.list_objects(test_bucket, prefix=f"{gold_prefix_root}/", recursive=True)
    if obj.object_name.startswith(f"{gold_prefix_root}/") and len(obj.object_name.split("/")) >= 4
})

# Error if no gold runs found
if not run_tags:
    raise RuntimeError(f"No gold runs found under: {gold_prefix_root}")

# Use the latest gold run for training
latest_gold_run_tag = run_tags[-1]
latest_gold_run_prefix = f"{gold_prefix_root}/{latest_gold_run_tag}"

# Save this for model artifact traceability
training_data_run_tag = latest_gold_run_tag

# Paths for train, valid and test files
# Path for label mapping file
train_path = f"{latest_gold_run_prefix}/train.parquet"
valid_path = f"{latest_gold_run_prefix}/valid.parquet"
test_path = f"{latest_gold_run_prefix}/test.parquet"
label_mapping_path = f"{latest_gold_run_prefix}/label_mapping.json"

# Function to load parquet files
def load_parquet_from_storage(bucket: str, object_name: str) -> pd.DataFrame:
    resp = client.get_object(bucket, object_name)
    try:
        return pd.read_parquet(io.BytesIO(resp.read()))
    finally:
        resp.close()
        resp.release_conn()

# Function to load json files
def load_json_from_storage(bucket: str, object_name: str) -> dict:
    resp = client.get_object(bucket, object_name)
    try:
        return json.loads(resp.read().decode("utf-8"))
    finally:
        resp.close()
        resp.release_conn()

# Files to dfs
train_df = load_parquet_from_storage(test_bucket, train_path)
valid_df = load_parquet_from_storage(test_bucket, valid_path)
test_df = load_parquet_from_storage(test_bucket, test_path)
label_mapping = load_json_from_storage(test_bucket, label_mapping_path)

# Print summary info on the run and loaded data
print("Latest gold run tag:", latest_gold_run_tag)
print("Latest gold prefix:", latest_gold_run_prefix)
print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows:", len(test_df))
print("Label mapping size:", len(label_mapping["classes"]))


Latest gold run tag: 20260308_205827
Latest gold prefix: gold/training/20260308_205827
Train rows: 7999
Valid rows: 1001
Test rows: 1000
Label mapping size: 12


In [20]:
# Label mapping check
display(label_mapping["classes"])

['App Support - ERP',
 'App Support - Finance',
 'App Support - HRIS',
 'App Support - M365',
 'App Support - Microsoft Fabric',
 'App Support - Power BI',
 'App Support - Power Platform',
 'End User Compute',
 'Identity and User Access',
 'Integration & Middleware',
 'Network Ops',
 'Security Operations']

# Validate data

In [21]:
# validate columns are present in all splits and check for leakage
# required cols
required_cols = ["sys_id", "text", "label_final", "label_id"]

# check for required cols in each split
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

    # Print now number, null rate, empty rate and number of labels in each split
    print(f"\n{name.upper()} checks")
    print("Rows:", len(df))
    print("Empty text rate:", f"{df['text'].fillna('').astype(str).str.strip().eq('').mean():.2%}")
    print("Null label rate:", f"{df['label_final'].isna().mean():.2%}")
    print("Unique labels:", df["label_final"].nunique())

# Collect sys_ids for leakage check
train_ids = set(train_df["sys_id"])
valid_ids = set(valid_df["sys_id"])
test_ids = set(test_df["sys_id"])

# Print overlap of sys_ids between splits to check for leakage
print("\nSplit leakage")
print("Train/Valid overlap:", len(train_ids & valid_ids))
print("Train/Test overlap :", len(train_ids & test_ids))
print("Valid/Test overlap :", len(valid_ids & test_ids))


TRAIN checks
Rows: 7999
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

VALID checks
Rows: 1001
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

TEST checks
Rows: 1000
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

Split leakage
Train/Valid overlap: 0
Train/Test overlap : 0
Valid/Test overlap : 0


# Functions for model development and iteration
Helper functions live together here so the run flow below stays simple: setup, build, train, evaluate, save.

In [22]:
# Turn the pandas splits into datasets for hugging face trainer
def prepare_hf_datasets(train_df, valid_df, test_df):
    train_view = train_df[["sys_id", "text", "label_id", "label_final"]].copy()
    valid_view = valid_df[["sys_id", "text", "label_id", "label_final"]].copy()
    test_view = test_df[["sys_id", "text", "label_id", "label_final"]].copy()

    train_view = train_view.rename(columns={"label_id": "labels"})
    valid_view = valid_view.rename(columns={"label_id": "labels"})
    test_view = test_view.rename(columns={"label_id": "labels"})

    hf_train = Dataset.from_pandas(train_view, preserve_index=False)
    hf_valid = Dataset.from_pandas(valid_view, preserve_index=False)
    hf_test = Dataset.from_pandas(test_view, preserve_index=False)

    return hf_train, hf_valid, hf_test

In [23]:
# Hugging face trainer needs label to id and id to label mappings 
def prepare_label_maps(label_mapping):
    label_to_id = {k: int(v) for k, v in label_mapping["label_to_id"].items()}
    id_to_label = {int(k): v for k, v in label_mapping["id_to_label"].items()}
    label_names_in_order = [id_to_label[i] for i in sorted(id_to_label.keys())]

    return label_to_id, id_to_label, label_names_in_order

In [24]:
# Rebuild a clean tokeniser and model init function for each run to ensure no cross run contamination
def build_tokeniser_and_model(checkpoint, label_mapping):
    label_to_id, id_to_label, _ = prepare_label_maps(label_mapping)

    tokeniser = AutoTokenizer.from_pretrained(checkpoint)

    def model_init():
        return AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=len(label_mapping["classes"]),
            id2label=id_to_label,
            label2id=label_to_id,
        )

    return tokeniser, model_init


In [25]:
# Tokenise every split the same way so train, validation and test stay aligned
def tokenise_splits(hf_train, hf_valid, hf_test, tokenizer, max_length):
    def tokenise_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length
        )

    tokenised_train = hf_train.map(tokenise_batch, batched=True)
    tokenised_valid = hf_valid.map(tokenise_batch, batched=True)
    tokenised_test = hf_test.map(tokenise_batch, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    return tokenised_train, tokenised_valid, tokenised_test, data_collator

In [26]:
# Keep the evaluation metrics simple and consistent across runs
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [27]:
# Build a fresh set of trainer arguments for each run
def build_training_args(
    run_name,
    output_root="./artifacts/c2_runs",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    max_steps=-1,
    weight_decay=0.01,
    warmup_ratio=0.0,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,
    metric_for_best_model=None,
    greater_is_better=True,
    logging_steps=20,
    seed=42,
    report_to="none",
):
    output_dir = str(Path(output_root) / run_name)

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        num_train_epochs=num_train_epochs,
        max_steps=max_steps,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        eval_strategy=eval_strategy,
        save_strategy=save_strategy,
        load_best_model_at_end=load_best_model_at_end,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=greater_is_better,
        save_total_limit=1,
        seed=seed,
        logging_steps=logging_steps,
        report_to=report_to,
    )

    return training_args


In [28]:
# Print the main numbers after a run
def summarise_run(result):
    print(f"Run name: {result['run_name']}")
    print(f"Checkpoint: {result['checkpoint']}")
    print(f"Output dir: {result['output_dir']}")
    print(f"Global steps: {result['global_steps']}")
    print(f"Training time (mins): {result['training_time_seconds'] / 60:.2f}")
    print(f"Validation accuracy: {result['validation_accuracy']:.4f}")
    print(f"Validation macro F1: {result['validation_macro_f1']:.4f}")
    print(f"Average max probability: {result['avg_max_probability']:.4f}")
    print(f"Manual review rate @ 0.70: {result['manual_review_rate']:.4f}")
    print(f"Lowest confidence: {result['lowest_confidence']:.4f}")
    print(f"Highest confidence: {result['highest_confidence']:.4f}")

In [29]:
# Helper functions to keep repeated runs clean and easy to reload in different session and/or notebooks
import gc
import shutil

# Paths for saving outputs and registry of runs
ARTIFACTS_ROOT = Path("./artifacts")
RUNS_ROOT = ARTIFACTS_ROOT / "c2_runs"
RUN_REGISTRY_PATH = ARTIFACTS_ROOT / "c2_run_registry.parquet"

# Function to clear memory between runs to reduce risk of contamination
def reset_training_runtime():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Function to prepare clean paths for a run and remove any data from previous runs
def prepare_run_registry_paths(run_name):
    run_dir = RUNS_ROOT / run_name

    if run_dir.exists():
        shutil.rmtree(run_dir)

    run_dir.mkdir(parents=True, exist_ok=True)

    return {
        "run_dir": run_dir,
        "metrics_json": run_dir / "metrics.json",
        "classification_report_txt": run_dir / "classification_report.txt",
        "confusion_matrix_csv": run_dir / "confusion_matrix.csv",
    }

# Function to build a predidction dataframe 
def build_prediction_frame(hf_dataset, prediction_output, id_to_label, confidence_threshold):
    logits = prediction_output.predictions
    labels = prediction_output.label_ids
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    max_probs = probs.max(axis=1)

    return pd.DataFrame({
        "sys_id": hf_dataset["sys_id"],
        "actual_label_id": labels,
        "actual_label": [id_to_label[int(label)] for label in labels],
        "predicted_label_id": preds,
        "predicted_label": [id_to_label[int(pred)] for pred in preds],
        "max_probability": max_probs,
        "needs_manual_review": max_probs < confidence_threshold,
    })

# Function to save the run outputs to files for analysis across sessions/notebooks *training time is long
def save_run_outputs(run_paths, metrics_payload, classification_report_text, confusion_matrix_df):
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    run_paths["metrics_json"].write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
    run_paths["classification_report_txt"].write_text(classification_report_text, encoding="utf-8")
    confusion_matrix_df.to_csv(run_paths["confusion_matrix_csv"], index=True)

# Function to update the run registry with a new summary overwriting the last run
def upsert_run_registry(summary_row, registry_path=RUN_REGISTRY_PATH):
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    incoming = pd.DataFrame([summary_row])

    if registry_path.exists():
        existing = pd.read_parquet(registry_path)
        existing = existing[existing["run_name"] != summary_row["run_name"]]
        incoming = pd.concat([existing, incoming], ignore_index=True)

    incoming = incoming.sort_values("run_name").reset_index(drop=True)
    incoming.to_parquet(registry_path, index=False)
    return incoming


# Implementation candidate #2
DistilBERT challenger model

Candidate 1 saw good performance overall, especially on specialised classes that had consistent distinct wording. It did however struggle on broader classes that have overlap with one another in terms of the wording or terminology often used to describe problems. 

Candidate 2 should provide better performance on classes with overlaps as it utilises a transformer based model which is designed to handle these types of challenges. Unlike TF-IDF which mainly looks at word frequency and short combinations, DistilBERT is able to look at relationships between words in context. In theory, this should translate to a better understanding that two incidents can mean the same thing even if they are worded differently by the caller.

DistilBERT was the first choice due to it's smaller and faster nature when comapred with the full version of BERT, this should give us a good balance between model capability and computational cost.

At this stage, the model will be un-tuned, taking default configuration from hugging face before we iterate with optimisation. The goal is to establish whether the transformer model itself can provide a meaningful improvement over the baseline before spending time tuning it further.

# C2_baseline

In [ ]:
# c2_baseline configuration 
RUN_NAME = "c2_baseline"
CHECKPOINT = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 128 # Shorter max length for faster training (limits context)
CONFIDENCE_THRESHOLD = 0.70 # Route anything for manual review under this level
LEARNING_RATE = 2e-5 # HF suggested starting point
PER_DEVICE_TRAIN_BATCH_SIZE = 16 # Max for hardware I have available
PER_DEVICE_EVAL_BATCH_SIZE = 16 # Max for hardware I have available
NUM_TRAIN_EPOCHS = 2 # Training over 2 epoch
MAX_STEPS = -1 # No max steps, train for full epochs
WEIGHT_DECAY = 0.01 # HF suggested starting point
WARMUP_RATIO = 0.0 # HF suggested starting point
EVAL_STRATEGY = "epoch" # Evaluate at end of each epoch
SAVE_STRATEGY = "epoch" # Save at end of each epoch
LOAD_BEST_MODEL_AT_END = True # Load best model at the end of the run
METRIC_FOR_BEST_MODEL = "macro_f1" # Use macro f1 to select best model
GREATER_IS_BETTER = True # High is better for model eval
LOGGING_STEPS = 20 # Log training metrics every 20 steps
SEED = 42 # Random seed for reproducibility

# Prepare datasets, label mappings and run paths for the training loop
hf_train, hf_valid, hf_test = prepare_hf_datasets(train_df, valid_df, test_df)
_, id_to_label, label_names_in_order = prepare_label_maps(label_mapping)
run_paths = prepare_run_registry_paths(RUN_NAME)

print("HF train rows:", len(hf_train))
print("HF valid rows:", len(hf_valid))
print("HF test rows :", len(hf_test))
print("Run name     :", RUN_NAME)


HF train rows: 7999
HF valid rows: 1001
HF test rows : 1000
Run name     : c2_baseline


In [ ]:
# Build a clean trainer for this run
reset_training_runtime()

# Fresh objects here stop one run leaking into the next one
tokenizer, model_init = build_tokeniser_and_model(CHECKPOINT, label_mapping)
tokenized_train, tokenized_valid, tokenized_test, data_collator = tokenise_splits(
    hf_train,
    hf_valid,
    hf_test,
    tokenizer,
    max_length=MAX_LENGTH,
)

# Build training arguments for this run, removes risk of accidently using from previous run
training_args = build_training_args(
    run_name=RUN_NAME,
    output_root=str(RUNS_ROOT),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,
    logging_steps=LOGGING_STEPS,
    seed=SEED,
)

# Build the trainer for this run
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer is ready for:", RUN_NAME)


Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer is ready for: c2_baseline


In [52]:
# Train the model and save the local run metrics and artefacts
train_start_time = time.time()
train_result = trainer.train()
training_time_seconds = time.time() - train_start_time

trainer.save_model(str(run_paths["run_dir"]))
tokenizer.save_pretrained(run_paths["run_dir"])

print(f"Training finished in {training_time_seconds / 60:.2f} minutes")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loade

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.445875,0.476835,0.839161,0.874454
2,0.502850,0.461235,0.839161,0.874454


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished in 54.03 minutes


In [54]:
# Evaluate on validation only for this comparison stage
# Use predict here so we do not depend on the notebook evaluate callback state
validation_output = trainer.predict(tokenized_valid, metric_key_prefix="validation")
validation_metrics = validation_output.metrics

# Build a dataframe of predictions, actuals for labels, max probabilities and if each row needs a manual review based on confidence
validation_predictions_df = build_prediction_frame(
    hf_dataset=hf_valid,
    prediction_output=validation_output,
    id_to_label=id_to_label,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

# Builda classification report
validation_preds = validation_predictions_df["predicted_label_id"].to_numpy()
validation_labels = validation_output.label_ids
classification_report_text = classification_report(
    validation_labels,
    validation_preds,
    target_names=label_names_in_order,
    digits=4,
)

# Build a confusion matrix
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(
        validation_labels,
        validation_preds,
        labels=list(sorted(id_to_label.keys())),
    ),
    index=label_names_in_order,
    columns=label_names_in_order,
)

# Prepare a summary of the run and save all outputs to local files
eval_strategy_value = training_args.eval_strategy.value if hasattr(training_args.eval_strategy, "value") else training_args.eval_strategy
save_strategy_value = training_args.save_strategy.value if hasattr(training_args.save_strategy, "value") else training_args.save_strategy

summary_row = {
    "run_name": RUN_NAME,
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_version": training_data_run_tag,
    "checkpoint": CHECKPOINT,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "eval_strategy": eval_strategy_value,
    "save_strategy": save_strategy_value,
    "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
    "global_steps": trainer.state.global_step,
    "training_time_seconds": training_time_seconds,
    "validation_accuracy": float(validation_metrics["validation_accuracy"]),
    "validation_macro_f1": float(validation_metrics["validation_macro_f1"]),
    "avg_max_probability": float(validation_predictions_df["max_probability"].mean()),
    "manual_review_rate": float(validation_predictions_df["needs_manual_review"].mean()),
    "lowest_confidence": float(validation_predictions_df["max_probability"].min()),
    "highest_confidence": float(validation_predictions_df["max_probability"].max()),
    "seed": SEED,
    "train_loss": float(train_result.training_loss),
    "validation_loss": float(validation_metrics["validation_loss"]),
}

# Upsert this run into the registry of runs and save all to local files
metrics_payload = dict(summary_row)
metrics_payload["output_dir"] = str(run_paths["run_dir"])
metrics_payload["best_checkpoint"] = trainer.state.best_model_checkpoint

# Run specific outputs
c2_baseline = {
    "run_name": RUN_NAME,
    "checkpoint": CHECKPOINT,
    "output_dir": str(run_paths["run_dir"]),
    "global_steps": trainer.state.global_step,
    "training_time_seconds": training_time_seconds,
    "validation_accuracy": float(validation_metrics["validation_accuracy"]),
    "validation_macro_f1": float(validation_metrics["validation_macro_f1"]),
    "avg_max_probability": float(validation_predictions_df["max_probability"].mean()),
    "manual_review_rate": float(validation_predictions_df["needs_manual_review"].mean()),
    "lowest_confidence": float(validation_predictions_df["max_probability"].min()),
    "highest_confidence": float(validation_predictions_df["max_probability"].max()),
    "classification_report": classification_report_text,
    "confusion_matrix_df": confusion_matrix_df,
}
# Print the summary
summarise_run(c2_baseline)


c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Run name: c2_baseline
Checkpoint: distilbert/distilbert-base-uncased
Output dir: artifacts\c2_runs\c2_baseline
Global steps: 1000
Training time (mins): 54.03
Validation accuracy: 0.8392
Validation macro F1: 0.8745
Average max probability: 0.8294
Manual review rate @ 0.70: 0.1888
Lowest confidence: 0.1702
Highest confidence: 0.9841


In [55]:
# Save the run artefacts and upsert the parquet registry
save_run_outputs(
    run_paths=run_paths,
    metrics_payload=metrics_payload,
    classification_report_text=classification_report_text,
    confusion_matrix_df=confusion_matrix_df,
)

registry_df = upsert_run_registry(summary_row)
registry_df[registry_df["run_name"] == RUN_NAME]


,run_name,run_timestamp_utc,dataset_version,checkpoint,max_length,learning_rate,per_device_train_batch_size,per_device_eval_batch_size,num_train_epochs,max_steps,...,training_time_seconds,validation_accuracy,validation_macro_f1,avg_max_probability,manual_review_rate,lowest_confidence,highest_confidence,seed,train_loss,validation_loss
0,c2_baseline,2026-03-14T12:32:53.071569+00:00,20260308_205827,distilbert/distilbert-base-uncased,128,0.00002,16,16,2,-1,...,3241.729228,0.839161,0.874454,0.829449,0.188811,0.170236,0.984061,42,0.742991,0.476833


In [56]:
# Quick review outputs for the current run
print(c2_baseline["classification_report"])
display(c2_baseline["confusion_matrix_df"])
pd.read_parquet(RUN_REGISTRY_PATH).query("run_name == @RUN_NAME")


                                precision    recall  f1-score   support

             App Support - ERP     1.0000    0.8293    0.9067        41
         App Support - Finance     1.0000    0.8158    0.8986        38
            App Support - HRIS     1.0000    0.8824    0.9375        34
            App Support - M365     0.5223    1.0000    0.6862       176
App Support - Microsoft Fabric     1.0000    0.8193    0.9007        83
        App Support - Power BI     1.0000    0.8125    0.8966       112
  App Support - Power Platform     1.0000    0.8252    0.9043       103
              End User Compute     1.0000    0.7857    0.8800        98
      Identity and User Access     1.0000    0.7126    0.8322        87
      Integration & Middleware     1.0000    0.7273    0.8421        33
                   Network Ops     1.0000    0.8284    0.9061       134
           Security Operations     1.0000    0.8226    0.9027        62

                      accuracy                         0.8392 

,App Support - ERP,App Support - Finance,App Support - HRIS,App Support - M365,App Support - Microsoft Fabric,App Support - Power BI,App Support - Power Platform,End User Compute,Identity and User Access,Integration & Middleware,Network Ops,Security Operations
App Support - ERP,34,0,0,7,0,0,0,0,0,0,0,0
App Support - Finance,0,31,0,7,0,0,0,0,0,0,0,0
App Support - HRIS,0,0,30,4,0,0,0,0,0,0,0,0
App Support - M365,0,0,0,176,0,0,0,0,0,0,0,0
App Support - Microsoft Fabric,0,0,0,15,68,0,0,0,0,0,0,0
App Support - Power BI,0,0,0,21,0,91,0,0,0,0,0,0
App Support - Power Platform,0,0,0,18,0,0,85,0,0,0,0,0
End User Compute,0,0,0,21,0,0,0,77,0,0,0,0
Identity and User Access,0,0,0,25,0,0,0,0,62,0,0,0
Integration & Middleware,0,0,0,9,0,0,0,0,0,24,0,0


,run_name,run_timestamp_utc,dataset_version,checkpoint,max_length,learning_rate,per_device_train_batch_size,per_device_eval_batch_size,num_train_epochs,max_steps,...,training_time_seconds,validation_accuracy,validation_macro_f1,avg_max_probability,manual_review_rate,lowest_confidence,highest_confidence,seed,train_loss,validation_loss
0,c2_baseline,2026-03-14T12:32:53.071569+00:00,20260308_205827,distilbert/distilbert-base-uncased,128,0.00002,16,16,2,-1,...,3241.729228,0.839161,0.874454,0.829449,0.188811,0.170236,0.984061,42,0.742991,0.476833


#### C2 Baseline evaluation:
Overall performance is good rather than exceptional as an increase over the linear model. Accuracy of 0.8392 and a macro F1 of 0.8745 show the model is viable and has potential for further improvement with some tuninig.

The model is very strong on the more distinct classes like ERP, Finance, HRIS, Fabric, Power BI, Power Platform and security operations. All of these classes have very high precision and recall, which tells us the model is good at identifying these classes and not misclassifying as other classes.

The main weakness is App Support - M365... it has perfect recall (1.0000) but really poor precision (0.5223). This means the model is sending too many incidents to this class that do not belong there. That basically makes this class the default option chosen by the model when it is unsure. This is likely due to the fact that this class is the largest in the dataset and has a lot of overlap with other classes in terms of wording. The model is likely defaulting to this class when it is unsure, which is why it has perfect recall but poor precision.

The validation accuracy and macro f1 are both the same over epoch 1 and 2, while validation loss improves slightly. This suggests the second epoch did not improve the decision boundary of the model, but it did make the model a bit more confident in its predictions. We should not immediately increase epochs as this could lead to overfitting and we have only seen marginal improvement in loss.

The confidence summary shows us the average max probability is 0.8294 which is quite high, with only an 18.9% manual review rate with the current 0.70 threshold. This is good as we are essentially able to automate the majority of incident allocation with a good level of confidence.

_____________________________________________________________________________________________________________________________

# C2_V1

The focus for c2_v1 is going to be to address the class imbalance issue by weighting the loss by class frequency. This should help to reduce the bias towards the largest class (M365), improve balance across other classes and reduce falase positives into the M365 class.

I'll aim to do this by:
Using class weighted cross entropy loss function, to penalise misclassification of the minority classes more than the majority class. This means:
- classes with less training examples get a higher weight
- classes with more training examples get a lower weight
- the model is penalised more when it misclassifies minority classes
- the model is penalised less for errors on the largest classes

This should lead to:
- M365 precision improvement as less incidents are misclassified into this class
- Recall for under represented classes should improve as the model is now more sensitive to these
- Macro F1 should improve as the balance between precision and recall improves across classes, especially for the minority ones.
- Overall accuracy may or may not improve
- M365 recall may drop a bit as the model becomes less willing to label edge cases into this class
- Manual review rate may increase a bit as the model becomes less confident in labelling edge cases into M365

In [30]:
# Imports needed for v1

from sklearn.utils.class_weight import compute_class_weight

In [31]:
# Helper function to compute balanced class weights for the training data
def build_balanced_class_weights(train_df, label_mapping):
    label_to_id, id_to_label, _ = prepare_label_maps(label_mapping)

    class_ids = np.array(sorted(train_df["label_id"].unique()))
    expected_class_ids = np.array(sorted(id_to_label.keys()))

    if not np.array_equal(class_ids, expected_class_ids):
        raise ValueError(
            "Training split does not contain the full expected label set. "
            f"Found {class_ids.tolist()}, expected {expected_class_ids.tolist()}."
        )

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=class_ids,
        y=train_df["label_id"].to_numpy(),
    )

    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float32)

    class_weight_df = pd.DataFrame({
        "label_id": class_ids,
        "label_name": [id_to_label[int(i)] for i in class_ids],
        "class_weight": class_weights,
    }).sort_values("class_weight", ascending=False).reset_index(drop=True)

    return class_weight_tensor, class_weight_df

In [32]:
# Custom trainer that applies class weights to the loss functon during training to help with class imbalance
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**model_inputs)
        logits = outputs.get("logits")

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [33]:
# c2_v1 config
RUN_NAME = "c2_v1"
CHECKPOINT = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 128
CONFIDENCE_THRESHOLD = 0.70
LEARNING_RATE = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 2
MAX_STEPS = -1
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.0
EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"
LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST_MODEL = "macro_f1"
GREATER_IS_BETTER = True
LOGGING_STEPS = 20
SEED = 42

In [34]:
# Prepare datasets, label mappings and run paths
hf_train, hf_valid, hf_test = prepare_hf_datasets(train_df, valid_df, test_df)
_, id_to_label, label_names_in_order = prepare_label_maps(label_mapping)
run_paths = prepare_run_registry_paths(RUN_NAME)

print("HF train rows:", len(hf_train))
print("HF valid rows:", len(hf_valid))
print("HF test rows :", len(hf_test))
print("Run name     :", RUN_NAME)

HF train rows: 7999
HF valid rows: 1001
HF test rows : 1000
Run name     : c2_v1


In [35]:
# Build class weights from the training split only
class_weight_tensor, class_weight_df = build_balanced_class_weights(train_df, label_mapping)
display(class_weight_df)

,label_id,label_name,class_weight
0,9,Integration & Middleware,2.544211
1,2,App Support - HRIS,2.415157
2,1,App Support - Finance,2.192708
3,0,App Support - ERP,2.051026
4,11,Security Operations,1.354844
5,4,App Support - Microsoft Fabric,0.999375
6,8,Identity and User Access,0.952262
7,7,End User Compute,0.854594
8,6,App Support - Power Platform,0.812907
9,5,App Support - Power BI,0.747291


In [36]:
# Build a clean trainer for this run
reset_training_runtime()

In [37]:
# Fresh objects to prevent leaking between runs
tokenizer, model_init = build_tokeniser_and_model(CHECKPOINT, label_mapping)
tokenized_train, tokenized_valid, tokenized_test, data_collator = tokenise_splits(
    hf_train,
    hf_valid,
    hf_test,
    tokenizer,
    max_length=MAX_LENGTH,
)

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [38]:
# Training arguments for the run
training_args = build_training_args(
    run_name=RUN_NAME,
    output_root=str(RUNS_ROOT),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,
    logging_steps=LOGGING_STEPS,
    seed=SEED,
)

# Build the train for the run using the custom trainer for class weighted loss
trainer = WeightedLossTrainer(
    class_weights=class_weight_tensor,
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer is ready for:", RUN_NAME)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer is ready for: c2_v1


In [39]:
train_start_time = time.time()
train_result = trainer.train()
training_time_seconds = time.time() - train_start_time

trainer.save_model(str(run_paths["run_dir"]))
tokenizer.save_pretrained(run_paths["run_dir"])

print(f"Training finished in {training_time_seconds / 60:.2f} minutes")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loade

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.428657,0.498709,0.839161,0.874454
2,0.543716,0.489966,0.839161,0.874454


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished in 51.31 minutes


In [41]:
# Validation evaluation
validation_output = trainer.predict(tokenized_valid, metric_key_prefix="validation")
validation_metrics = validation_output.metrics

validation_predictions_df = build_prediction_frame(
    hf_dataset=hf_valid,
    prediction_output=validation_output,
    id_to_label=id_to_label,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

validation_preds = validation_predictions_df["predicted_label_id"].to_numpy()
validation_labels = validation_output.label_ids

classification_report_text = classification_report(
    validation_labels,
    validation_preds,
    target_names=label_names_in_order,
    digits=4,
)

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(
        validation_labels,
        validation_preds,
        labels=list(sorted(id_to_label.keys())),
    ),
    index=label_names_in_order,
    columns=label_names_in_order,
)

eval_strategy_value = (
    training_args.eval_strategy.value
    if hasattr(training_args.eval_strategy, "value")
    else training_args.eval_strategy
)
save_strategy_value = (
    training_args.save_strategy.value
    if hasattr(training_args.save_strategy, "value")
    else training_args.save_strategy
)

summary_row = {
    "run_name": RUN_NAME,
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_version": training_data_run_tag,
    "checkpoint": CHECKPOINT,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "eval_strategy": eval_strategy_value,
    "save_strategy": save_strategy_value,
    "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
    "global_steps": trainer.state.global_step,
    "training_time_seconds": training_time_seconds,
    "validation_accuracy": float(validation_metrics["validation_accuracy"]),
    "validation_macro_f1": float(validation_metrics["validation_macro_f1"]),
    "avg_max_probability": float(validation_predictions_df["max_probability"].mean()),
    "manual_review_rate": float(validation_predictions_df["needs_manual_review"].mean()),
    "lowest_confidence": float(validation_predictions_df["max_probability"].min()),
    "highest_confidence": float(validation_predictions_df["max_probability"].max()),
    "seed": SEED,
    "train_loss": float(train_result.training_loss),
    "validation_loss": float(validation_metrics["validation_loss"]),
    "loss_strategy": "class_weighted_cross_entropy",
    "class_weight_mode": "balanced",
}

metrics_payload = dict(summary_row)
metrics_payload["output_dir"] = str(run_paths["run_dir"])
metrics_payload["best_checkpoint"] = trainer.state.best_model_checkpoint
metrics_payload["class_weights"] = (
    class_weight_df.sort_values("label_id")
    .set_index("label_name")["class_weight"]
    .to_dict()
)

c2_v1 = {
    "run_name": RUN_NAME,
    "checkpoint": CHECKPOINT,
    "output_dir": str(run_paths["run_dir"]),
    "global_steps": trainer.state.global_step,
    "training_time_seconds": training_time_seconds,
    "validation_accuracy": float(validation_metrics["validation_accuracy"]),
    "validation_macro_f1": float(validation_metrics["validation_macro_f1"]),
    "avg_max_probability": float(validation_predictions_df["max_probability"].mean()),
    "manual_review_rate": float(validation_predictions_df["needs_manual_review"].mean()),
    "lowest_confidence": float(validation_predictions_df["max_probability"].min()),
    "highest_confidence": float(validation_predictions_df["max_probability"].max()),
    "classification_report": classification_report_text,
    "confusion_matrix_df": confusion_matrix_df,
}

summarise_run(c2_v1)

c:\Users\JHoll\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Run name: c2_v1
Checkpoint: distilbert/distilbert-base-uncased
Output dir: artifacts\c2_runs\c2_v1
Global steps: 1000
Training time (mins): 51.31
Validation accuracy: 0.8392
Validation macro F1: 0.8745
Average max probability: 0.8130
Manual review rate @ 0.70: 0.1888
Lowest confidence: 0.1067
Highest confidence: 0.9807


In [42]:
save_run_outputs(
    run_paths=run_paths,
    metrics_payload=metrics_payload,
    classification_report_text=classification_report_text,
    confusion_matrix_df=confusion_matrix_df,
)

registry_df = upsert_run_registry(summary_row)
display(registry_df[registry_df["run_name"] == RUN_NAME])

print(c2_v1["classification_report"])
display(c2_v1["confusion_matrix_df"])

,run_name,run_timestamp_utc,dataset_version,checkpoint,max_length,learning_rate,per_device_train_batch_size,per_device_eval_batch_size,num_train_epochs,max_steps,...,validation_macro_f1,avg_max_probability,manual_review_rate,lowest_confidence,highest_confidence,seed,train_loss,validation_loss,loss_strategy,class_weight_mode
1,c2_v1,2026-03-14T16:26:55.295705+00:00,20260308_205827,distilbert/distilbert-base-uncased,128,0.00002,16,16,2,-1,...,0.874454,0.812998,0.188811,0.106724,0.98071,42,0.781122,0.498709,class_weighted_cross_entropy,balanced


                                precision    recall  f1-score   support

             App Support - ERP     1.0000    0.8293    0.9067        41
         App Support - Finance     1.0000    0.8158    0.8986        38
            App Support - HRIS     1.0000    0.8824    0.9375        34
            App Support - M365     0.5223    1.0000    0.6862       176
App Support - Microsoft Fabric     1.0000    0.8193    0.9007        83
        App Support - Power BI     1.0000    0.8125    0.8966       112
  App Support - Power Platform     1.0000    0.8252    0.9043       103
              End User Compute     1.0000    0.7857    0.8800        98
      Identity and User Access     1.0000    0.7126    0.8322        87
      Integration & Middleware     1.0000    0.7273    0.8421        33
                   Network Ops     1.0000    0.8284    0.9061       134
           Security Operations     1.0000    0.8226    0.9027        62

                      accuracy                         0.8392 

,App Support - ERP,App Support - Finance,App Support - HRIS,App Support - M365,App Support - Microsoft Fabric,App Support - Power BI,App Support - Power Platform,End User Compute,Identity and User Access,Integration & Middleware,Network Ops,Security Operations
App Support - ERP,34,0,0,7,0,0,0,0,0,0,0,0
App Support - Finance,0,31,0,7,0,0,0,0,0,0,0,0
App Support - HRIS,0,0,30,4,0,0,0,0,0,0,0,0
App Support - M365,0,0,0,176,0,0,0,0,0,0,0,0
App Support - Microsoft Fabric,0,0,0,15,68,0,0,0,0,0,0,0
App Support - Power BI,0,0,0,21,0,91,0,0,0,0,0,0
App Support - Power Platform,0,0,0,18,0,0,85,0,0,0,0,0
End User Compute,0,0,0,21,0,0,0,77,0,0,0,0
Identity and User Access,0,0,0,25,0,0,0,0,62,0,0,0
Integration & Middleware,0,0,0,9,0,0,0,0,0,24,0,0


In [43]:
print("Min class weight:", class_weight_df["class_weight"].min())
print("Max class weight:", class_weight_df["class_weight"].max())
print("Weight ratio max/min:", class_weight_df["class_weight"].max() / class_weight_df["class_weight"].min())

Min class weight: 0.47275413711583925
Max class weight: 2.544211195928753
Weight ratio max/min: 5.381679389312977


In [46]:
# Sanity check: prove weighted loss is different from unweighted loss
sample_items = []
for i in range(8):
    item = {
        "input_ids": tokenized_train[i]["input_ids"],
        "attention_mask": tokenized_train[i]["attention_mask"],
        "labels": tokenized_train[i]["labels"],
    }
    if "token_type_ids" in tokenized_train.column_names:
        item["token_type_ids"] = tokenized_train[i]["token_type_ids"]
    sample_items.append(item)

sample_batch = data_collator(sample_items)

model = model_init()
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=sample_batch["input_ids"],
        attention_mask=sample_batch["attention_mask"],
    )
    logits = outputs.logits
    labels = sample_batch["labels"]

    unweighted_loss = torch.nn.CrossEntropyLoss()(logits, labels)
    weighted_loss = torch.nn.CrossEntropyLoss(weight=class_weight_tensor)(logits, labels)

print("Unweighted loss:", float(unweighted_loss))
print("Weighted loss  :", float(weighted_loss))
print("Different?     :", float(unweighted_loss) != float(weighted_loss))
print("Batch labels   :", labels.tolist())

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Unweighted loss: 2.509064197540283
Weighted loss  : 2.519181728363037
Different?     : True
Batch labels   : [6, 6, 6, 4, 8, 9, 6, 4]
